In [ ]:
# @title Replace files in a Dataverse dataset
# Run this cell, enter your API token and DOI number, then upload the replacement files.

import os
import sys
import json
import hashlib
import subprocess
import requests
from getpass import getpass


# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================

def install_packages():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "pyDataverse",
            "-q"
        ]
    )


try:
    import pyDataverse
except ImportError:
    print("Installing required libraries...")
    install_packages()


from pyDataverse.api import NativeApi
from google.colab import files


# ============================================================
# USER INPUT
# ============================================================

# API token is hidden while typing
API_TOKEN = getpass("Please enter your API token: ")

# User only enters the numerical part:
# Example: doi:10.34810/data3724 -> enter 3724
doi_input = input(
    "Please enter the LAST DIGITS of the DOI "
    "(for example 3724): "
).strip()

identifier = f"doi:10.34810/data{doi_input}"

SERVER_URL = "https://dataverse.csuc.cat"


# ============================================================
# FORMATS THAT DATAVERSE MAY INGEST AS .tab
# ============================================================

TABULAR_EXTENSIONS = {
    ".xlsx",
    ".xls",
    ".csv",
    ".tsv",
    ".sav",
    ".por",
    ".dta",
    ".rdata",
    ".rda"
}


# ============================================================
# GET FILE METADATA FROM DATAVERSE
# ============================================================

def get_dataverse_files(server_url, api_token, doi):

    api = NativeApi(server_url, api_token)

    try:
        dataset = api.get_dataset(doi)

        if dataset.status_code != 200:

            try:
                response_json = dataset.json()

                message = (
                    response_json.get("message")
                    or response_json.get("data", {}).get("message")
                    or str(response_json)
                )

            except Exception:
                message = dataset.text

            print(
                f"❌ Error retrieving dataset {doi}: "
                f"{message}"
            )

            return []

        file_entries = (
            dataset.json()
            .get("data", {})
            .get("latestVersion", {})
            .get("files", [])
        )

        result = []

        for file_entry in file_entries:

            data_file = file_entry.get(
                "dataFile",
                {}
            ).copy()

            # File-level metadata that is outside dataFile
            data_file["description"] = file_entry.get(
                "description",
                ""
            )

            data_file["directoryLabel"] = file_entry.get(
                "directoryLabel",
                ""
            )

            data_file["categories"] = file_entry.get(
                "categories",
                []
            )

            result.append(data_file)

        return result

    except Exception as e:

        print(
            f"❌ Error retrieving dataset {doi}: {e}"
        )

        return []


# ============================================================
# FIND MATCHING DATAVERSE FILE
# ============================================================

def find_dataverse_file(
    local_filename,
    metadata_dict
):

    # 1. Exact filename match
    if local_filename in metadata_dict:

        return (
            metadata_dict[local_filename],
            "exact"
        )

    # 2. Check if Dataverse ingested it as .tab
    base_name, extension = os.path.splitext(
        local_filename
    )

    extension = extension.lower()

    if extension in TABULAR_EXTENSIONS:

        tab_filename = base_name + ".tab"

        if tab_filename in metadata_dict:

            return (
                metadata_dict[tab_filename],
                "ingested"
            )

    return None, None


# ============================================================
# CHECKSUM
# ============================================================

def compute_file_hash(
    file_path,
    algorithm
):

    algorithms = {
        "MD5": hashlib.md5,
        "SHA-1": hashlib.sha1,
        "SHA-256": hashlib.sha256,
        "SHA-512": hashlib.sha512
    }

    hash_function = algorithms.get(
        algorithm.upper()
    )

    if hash_function is None:
        return None

    hasher = hash_function()

    with open(file_path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


# ============================================================
# GET DATAVERSE FILE INFORMATION
# ============================================================

dataverse_files = get_dataverse_files(
    SERVER_URL,
    API_TOKEN,
    identifier
)


# Stop if dataset could not be retrieved
if not dataverse_files:

    raise SystemExit(
        "The dataset could not be retrieved "
        "or contains no files."
    )


# Create dictionary using filename as key
metadata_dict = {}

for metadata in dataverse_files:

    filename = metadata.get("filename")

    if filename:
        metadata_dict[filename] = metadata


# ============================================================
# UPLOAD REPLACEMENT FILES
# ============================================================

print()
print(
    "Select the file or files that you want "
    "to replace in Dataverse."
)

uploaded = files.upload()


if not uploaded:

    raise SystemExit(
        "No files were uploaded."
    )


# ============================================================
# PROCESS EACH UPLOADED FILE
# ============================================================

for local_filename in uploaded.keys():

    file_path = local_filename


    # --------------------------------------------------------
    # FIND MATCHING FILE
    # --------------------------------------------------------

    file_metadata, match_type = find_dataverse_file(
        local_filename,
        metadata_dict
    )


    if file_metadata is None:

        print(
            f"⚠️ Not replaced "
            f"(no matching file): "
            f"{local_filename}"
        )

        continue


    # --------------------------------------------------------
    # GET DATAVERSE FILE ID
    # --------------------------------------------------------

    file_id = file_metadata.get("id")


    if file_id is None:

        print(
            f"❌ Error replacing "
            f"{local_filename}: "
            f"Dataverse file ID not found."
        )

        continue


    # --------------------------------------------------------
    # CHECK IF FILE IS IDENTICAL
    #
    # Only for exact matches.
    #
    # For XLSX -> TAB, CSV -> TAB, etc.,
    # the checksum of the .tab is not the checksum
    # of the original uploaded file.
    # --------------------------------------------------------

    if match_type == "exact":

        checksum = file_metadata.get(
            "checksum",
            {}
        )

        checksum_type = checksum.get(
            "type"
        )

        existing_checksum = checksum.get(
            "value"
        )


        if checksum_type and existing_checksum:

            new_checksum = compute_file_hash(
                file_path,
                checksum_type
            )


            if (
                new_checksum
                and
                new_checksum.lower()
                == existing_checksum.lower()
            ):

                print(
                    f"⏭️ Not replaced "
                    f"(identical): "
                    f"{local_filename}"
                )

                continue


    # --------------------------------------------------------
    # FILE METADATA TO PRESERVE
    # --------------------------------------------------------

    json_data = {

        "description":
            file_metadata.get(
                "description",
                ""
            ),

        "directoryLabel":
            file_metadata.get(
                "directoryLabel",
                ""
            ),

        "categories":
            file_metadata.get(
                "categories",
                []
            ),

        "forceReplace": True
    }


    # --------------------------------------------------------
    # DATAVERSE REPLACE API
    # --------------------------------------------------------

    url = (
        f"{SERVER_URL}/api/files/"
        f"{file_id}/replace"
    )


    headers = {
        "X-Dataverse-key": API_TOKEN
    }


    # --------------------------------------------------------
    # REPLACE FILE
    # --------------------------------------------------------

    try:

        with open(
            file_path,
            "rb"
        ) as file:

            request_files = {

                "file": (
                    local_filename,
                    file
                )
            }


            request_data = {

                "jsonData":
                    json.dumps(
                        json_data
                    )
            }


            response = requests.post(

                url,

                headers=headers,

                files=request_files,

                data=request_data,

                timeout=300
            )


        # ----------------------------------------------------
        # READ RESPONSE
        # ----------------------------------------------------

        try:
            response_json = response.json()

        except ValueError:

            print(
                f"❌ Error replacing "
                f"{local_filename}: "
                f"HTTP {response.status_code} - "
                f"{response.text[:300]}"
            )

            continue


        # ----------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------

        if (
            response.status_code == 200
            and
            response_json.get("status") == "OK"
        ):

            print(
                f"✅ Replaced: "
                f"{local_filename}"
            )


        # ----------------------------------------------------
        # DATAVERSE ERROR
        # ----------------------------------------------------

        else:

            message = response_json.get(
                "message"
            )


            if not message:

                response_data = response_json.get(
                    "data",
                    {}
                )

                if isinstance(
                    response_data,
                    dict
                ):

                    message = response_data.get(
                        "message"
                    )


            if not message:

                message = str(
                    response_json
                )


            print(
                f"❌ Error replacing "
                f"{local_filename}: "
                f"{message}"
            )


    # --------------------------------------------------------
    # NETWORK ERROR
    # --------------------------------------------------------

    except requests.RequestException as e:

        print(
            f"❌ Error replacing "
            f"{local_filename}: "
            f"{e}"
        )


    except Exception as e:

        print(
            f"❌ Error replacing "
            f"{local_filename}: "
            f"{e}"
        )